# Etapa 1: Selección y caracterización del dataset NYC TLC Yellow Taxi

En esta etapa se caracteriza el dataset de viajes en taxi amarillo de la ciudad de Nueva York durante los años 2024 y 2025, publicado por la New York City Taxi and Limousine Commission (TLC). El dataset contiene registros transaccionales de viajes con atributos de fechas, distancias, tarifas y zonas de origen y destino, lo cual lo hace adecuado para el análisis con PySpark en un entorno de Big Data.

Se utilizan dos años completos (2024 y 2025) para garantizar un volumen total superior a 1 GB y para habilitar análisis comparativo interanual. El año 2026, parcialmente publicado al momento de este trabajo, se reserva como conjunto de validación temporal para las etapas posteriores del proyecto.

El notebook está pensado para ejecutarse de forma portable, tanto localmente como en Google Colab. Todas las rutas de datos son relativas al notebook (`./data/raw`), por lo que cualquier integrante del equipo puede clonar el repositorio y ejecutar las celdas sin ajustes adicionales.

## 1. Descarga de datos

Los archivos se obtienen directamente del CDN oficial del TLC en formato Parquet. Desde 2022 el TLC distribuye los registros de viajes en Parquet de manera nativa, por su mejor compresión y lectura columnar respecto a CSV. La descarga se realiza de forma idempotente: si el archivo ya existe en disco, se omite, lo cual permite re-ejecutar el notebook sin volver a bajar los datos.

In [ ]:
from pathlib import Path
import subprocess

CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
DATA_DIR = Path("data/raw")
YEARS = [2024, 2025]
LOOKUP_FILE = "taxi_zone_lookup.csv"

In [ ]:
def download_if_missing(download_url, target_path):
    """Descarga `download_url` a `target_path` solo si `target_path` no existe.

    Devuelve un string con el estado: 'skip', 'ok' o 'error: <mensaje>'.
    """
    if target_path.exists():
        return "skip"

    target_path.parent.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["curl", "-sSL", "-o", str(target_path), download_url],
        capture_output=True,
        timeout=900,
    )

    if result.returncode != 0:
        return f"error: curl exit {result.returncode}"

    return "ok"

In [ ]:
# Parquets mensuales de viajes
for year in YEARS:
    for month in range(1, 13):
        filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
        url = f"{CDN_BASE}/trip-data/{filename}"
        target = DATA_DIR / filename
        status = download_if_missing(url, target)
        print(f"{status:>6}  {filename}")

# Tabla de referencia de zonas de taxi
url = f"{CDN_BASE}/misc/{LOOKUP_FILE}"
target = DATA_DIR / LOOKUP_FILE
status = download_if_missing(url, target)
print(f"{status:>6}  {LOOKUP_FILE}")

## 2. Resumen de archivos descargados

Se reporta el inventario y el tamaño en disco. La rúbrica del curso pide un dataset por encima de 1 GB. El formato Parquet ya está comprimido, por lo que el tamaño en disco es menor al equivalente en CSV pero conserva la totalidad de los registros.

In [ ]:
files = sorted(DATA_DIR.glob("*"))
total_bytes = sum(f.stat().st_size for f in files)

for f in files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f"{size_mb:>8.1f} MB   {f.name}")

print()
print(f"Archivos: {len(files)} (esperados: {len(YEARS) * 12 + 1})")
print(f"Tamaño total: {total_bytes / (1024 ** 3):.2f} GB")

parquets = [f for f in files if f.suffix == ".parquet"]
if parquets:
    avg_mb = sum(f.stat().st_size for f in parquets) / len(parquets) / (1024 ** 2)
print(f"Tamaño promedio por Parquet: {avg_mb:.1f} MB")

## 3. Carga del dataset con PySpark

En esta sección se inicializa una sesión local de PySpark y se cargan los 24 archivos Parquet de viajes en un único DataFrame, pasando la lista explícita de rutas a `spark.read.parquet()`.

PySpark implementa evaluación diferida (lazy evaluation): las transformaciones se registran pero no se ejecutan hasta que se invoca una acción como `count()` o `show()`. Esto permite optimizar el plan de ejecución y distribuir el trabajo entre particiones. La carga real de datos ocurre cuando se ejecuta la primera acción, no cuando se define el DataFrame.

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()

# Sube el umbral del log de planes (default 25). Algunas agregaciones de la sección 5
# generan más de 25 expresiones (14 columnas con min y max = 28), lo que dispara un
# WARN benigno que solo trunca el plan en el log, no la computación.
spark.conf.set("spark.sql.debug.maxToStringFields", 100)

print(f"Spark version: {spark.version}")

### Lista explícita de archivos y unificación de esquemas

En lugar de pasar un patrón glob (`yellow_tripdata_*.parquet`), construimos la lista explícita de rutas con `Path.glob()` y la entregamos a `spark.read.parquet(*paths)`. Esto evita un warning interno que dispara el patrón glob al verificar la existencia de un directorio de metadatos de structured streaming, y deja el output del notebook limpio para el entregable.

Adicionalmente se habilita la opción `mergeSchema=True`. NYC TLC introdujo la columna `cbd_congestion_fee` a partir del 5 de enero de 2025 (cargo por la zona de descongestión central de Manhattan). Los archivos de 2024 no la incluyen, los de 2025 sí. Sin esta opción, Spark toma el esquema del primer archivo y descarta silenciosamente columnas que aparecen en archivos posteriores. Con `mergeSchema=True`, Spark inspecciona el esquema de todos los archivos y construye la unión: los registros de 2024 quedan con `cbd_congestion_fee = null` y los de 2025 con su valor real. Referencia: https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#schema-merging

In [ ]:
# Lista explícita de rutas a los 24 archivos mensuales
parquet_paths = sorted(str(p) for p in DATA_DIR.glob("yellow_tripdata_*.parquet"))

# mergeSchema=True para preservar cbd_congestion_fee (presente solo desde 2025-01-05)
df = spark.read.option("mergeSchema", "true").parquet(*parquet_paths)

print(f"Archivos cargados: {len(parquet_paths)}")

In [ ]:
df.printSchema()

### Introspección con printSchema()

El método `printSchema()` muestra el árbol completo de tipos del DataFrame, incluyendo:
- Nombre de cada columna
- Tipo de datos (StringType, LongType, DoubleType, TimestampType, etc.)
- Indicador de nullabilidad: `true` si la columna puede contener nulos, `false` si no.

Esto es más informativo que acceder a `df.columns` o `df.dtypes` directamente. Referencia: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.printSchema.html

In [ ]:
n_rows = df.count()

print(f"Total de filas: {n_rows:,}")

### Acciones y evaluación diferida

`count()` es una acción de Spark que dispara la evaluación completa del plan de ejecución. Durante la primera ejecución, Spark lee todos los archivos Parquet desde disco (aproximadamente 1.4 GB) y cuenta las filas. Este proceso puede tardar varios minutos en una máquina local. El resultado se imprime con formato de separador de miles para mejor legibilidad.

Es importante notar que Spark **no** memoriza resultados entre acciones por defecto. Si se ejecuta otra acción sobre `df` (otro `count()`, un `summary()`, un `groupBy().agg()`, etc.), Spark vuelve a leer los archivos Parquet desde disco. Para evitar este re-procesamiento se utiliza `df.cache()` o `df.persist()`, decisión que se evaluará en el paso siguiente cuando el DataFrame se reuse múltiples veces. Referencia: https://spark.apache.org/docs/latest/rdd-programming-guide.html#rdd-persistence

Por su parte, `df.rdd.getNumPartitions()` informa cuántas particiones físicas distribuyen el DataFrame. Este número depende del tamaño y cantidad de archivos Parquet leídos, y determina el grado de paralelismo de las operaciones posteriores.

In [ ]:
print(f"Particiones: {df.rdd.getNumPartitions()}")
print()

zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv"))
)
print("Tabla de referencia de zonas de taxi:")
zones.show(5)

### 3.1 Caracterización del catálogo de zonas

Aunque el archivo `taxi_zone_lookup.csv` pesa solo ~12 KB y no califica como Big Data por sí mismo, es la tabla de dimensiones que da semántica a `PULocationID` y `DOLocationID` en los 90 millones de viajes. Antes de avanzar conviene caracterizarlo: tamaño exacto, esquema (re-leído con `inferSchema=True` para que `LocationID` sea entero y no string), distribución por borough y nivel de servicio, presencia de nulos, y cobertura del rango esperado de IDs.

In [ ]:
from pyspark.sql import functions as F

# 3.1 Caracterización del catálogo de zonas
print(f"Total de zonas: {zones.count()}")
print()

print("Esquema (con inferSchema):")
zones.printSchema()

print("Distribución por Borough:")
zones.groupBy("Borough").count().orderBy(F.desc("count")).show(truncate=False)

print("Distribución por service_zone:")
zones.groupBy("service_zone").count().orderBy(F.desc("count")).show(truncate=False)

print("Conteo de nulos por columna:")
zones.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in zones.columns]).show()

print("Cobertura del rango de IDs (esperado 1 a 265):")
ids_present = {row.LocationID for row in zones.select("LocationID").collect()}
expected = set(range(1, 266))
missing = sorted(expected - ids_present)
extra = sorted(ids_present - expected)
print(f"  IDs ausentes en el catálogo: {missing if missing else 'ninguno'}")
print(f"  IDs fuera del rango 1-265:   {extra if extra else 'ninguno'}")

### Lectura del bloque 3.1

El catálogo cubre las zonas TLC referenciadas por `PULocationID` y `DOLocationID` en `df`. La columna `LocationID` es la clave primaria (debe ser única, sin nulos) y los IDs deben ir de 1 a 265 según el diccionario, donde 264 y 265 corresponden a "Unknown" y "Outside of NYC". La columna `service_zone` agrupa zonas por nivel de servicio (Yellow Zone, Boro Zone, Airports, EWR), lo cual será útil en etapas posteriores cuando se analicen patrones de tarifa o demanda por categoría operativa.

La tabla es lo bastante pequeña como para inspeccionarla por completo (no es un caso de Big Data por sí sola), pero su integridad es crítica: cualquier `PULocationID` o `DOLocationID` en `df` que no exista en este catálogo es un error referencial que se documentará en la etapa de calidad.

### Datos cargados

Quedan disponibles dos DataFrames distribuidos en la sesión de Spark:

- `df`: registros de viajes en taxi amarillo (2024-2025), tamaño en disco ~1.4 GB y ~89.9 millones de filas.
- `zones`: tabla de referencia de zonas de taxi (caracterizada en la sección 3.1).

A continuación se documenta el diccionario de variables (sección 4), se validan los rangos efectivos para justificar refinamiento de tipos (sección 5), y finalmente se aplican los downcasts en sitio sobre `df` (sección 6).

## 4. Diccionario de variables (NYC TLC)

La siguiente tabla documenta cada columna del dataset según el diccionario oficial publicado por la New York City Taxi and Limousine Commission. Esta documentación es la fuente de verdad para los rangos válidos, valores enumerados y semántica de cada campo, y se usará en la sección siguiente para justificar refinamientos de tipos y reglas de validación de calidad.

Referencia: https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf (versión 18 de marzo de 2025)

| Columna | Tipo en Parquet | Descripción | Rango / valores válidos |
|---|---|---|---|
| `VendorID` | integer | Código del proveedor TPEP que generó el registro | enum: 1=Creative Mobile Technologies, 2=Curb Mobility, 6=Myle Technologies, 7=Helix |
| `tpep_pickup_datetime` | timestamp_ntz | Fecha y hora cuando el taxímetro fue activado | dentro del año del archivo |
| `tpep_dropoff_datetime` | timestamp_ntz | Fecha y hora cuando el taxímetro fue desactivado | posterior a `tpep_pickup_datetime` |
| `passenger_count` | long | Número de pasajeros en el vehículo | entero ≥ 0; típicamente 1 a 6 |
| `trip_distance` | double | Distancia del viaje en millas reportada por el taxímetro | ≥ 0 millas |
| `RatecodeID` | long | Código de tarifa aplicado al final del viaje | enum: 1=Standard, 2=JFK, 3=Newark, 4=Nassau/Westchester, 5=Negotiated, 6=Group ride, 99=Null/unknown |
| `store_and_fwd_flag` | string | Indica si el registro se almacenó en memoria del vehículo antes de transmitirse al servidor | enum: 'Y', 'N' |
| `PULocationID` | integer | Zona TLC donde se inició el viaje | entero, generalmente 1 a 263 (más 264 y 265 para zonas desconocidas) |
| `DOLocationID` | integer | Zona TLC donde se finalizó el viaje | igual que `PULocationID` |
| `payment_type` | long | Método de pago | enum: 0=Flex Fare, 1=Credit card, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown, 6=Voided trip |
| `fare_amount` | double | Tarifa por tiempo y distancia calculada por el taxímetro (USD) | ≥ 0 USD |
| `extra` | double | Extras y recargos diversos (USD) | ≥ 0 USD |
| `mta_tax` | double | Impuesto MTA aplicado según la tarifa metered (USD) | ≥ 0 USD |
| `tip_amount` | double | Propina (solo se registra para pagos con tarjeta de crédito; las propinas en efectivo no aparecen) (USD) | ≥ 0 USD |
| `tolls_amount` | double | Suma total de peajes pagados durante el viaje (USD) | ≥ 0 USD |
| `improvement_surcharge` | double | Recargo de mejora aplicado al inicio del viaje (vigente desde 2015) (USD) | ≥ 0 USD |
| `total_amount` | double | Monto total cobrado al pasajero (no incluye propina en efectivo) (USD) | ≥ 0 USD |
| `congestion_surcharge` | double | Recargo por congestión del estado de NY (USD) | ≥ 0 USD |
| `Airport_fee` | double | Cargo solo para abordajes en LaGuardia y JFK (USD) | ≥ 0 USD |
| `cbd_congestion_fee` | double | Cargo por la zona de descongestión central de Manhattan, vigente desde 2025-01-05 (USD) | ≥ 0 USD; null para registros previos a 2025-01-05 |

## 5. Validación previa al refinamiento de tipos

El esquema actual del dataset está sobre-dimensionado en varias columnas: identificadores enumerados (`VendorID`, `RatecodeID`, `payment_type`) están almacenados como `long` (64 bits) cuando sus valores válidos caben en `byte` (8 bits); las zonas (`PULocationID`, `DOLocationID`) están en `integer` (32 bits) cuando caben en `short` (16 bits); y los montos monetarios están en `double` (64 bits) cuando para tarifas de taxi (típicamente menores a USD 1000) la precisión de `float` (32 bits) es suficiente.

Antes de aplicar cualquier downcast es necesario validar empíricamente que los valores reales en los datos caben en el tipo más estrecho. Esta sección ejecuta tres bloques de validación:

1. **Valores distintos en columnas enumeradas** para confirmar que coinciden con el diccionario oficial.
2. **Mínimos y máximos en columnas numéricas** para confirmar que los rangos efectivos caben en los tipos más estrechos propuestos.
3. **Presencia de la columna `cbd_congestion_fee`** y proporción de nulos por año, para verificar que `mergeSchema=True` la rescató correctamente.

Si las validaciones pasan, se aplicarán los casts en la sección 6 con justificación por columna. Si alguna validación falla, se documenta como hallazgo y se decide si tratarlo como dato sucio en la etapa de calidad o ampliar el tipo destino.

In [ ]:
# 5.1 Valores distintos en columnas enumeradas

enum_cols = ["VendorID", "RatecodeID", "payment_type", "store_and_fwd_flag"]

for col_name in enum_cols:
    print(f"\n{col_name}:")
    (df.groupBy(col_name)
        .count()
        .orderBy(col_name)
        .show(truncate=False))

### Lectura del bloque 5.1

Cada tabla muestra los valores únicos observados en una columna enumerada y su frecuencia. Se compara contra los valores documentados:

- `VendorID`: deben observarse subconjuntos de {1, 2, 6, 7}.
- `RatecodeID`: subconjuntos de {1, 2, 3, 4, 5, 6, 99}.
- `payment_type`: subconjuntos de {0, 1, 2, 3, 4, 5, 6}.
- `store_and_fwd_flag`: subconjuntos de {'Y', 'N'}.

La presencia de un valor `null` se cuenta como una categoría más y se documenta. Valores fuera del catálogo oficial se registrarán como problema de calidad en el paso siguiente.

In [ ]:
# 5.2 Rangos efectivos de columnas numéricas
numeric_cols = [
    "passenger_count", "trip_distance",
    "PULocationID", "DOLocationID",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge",
    "Airport_fee", "cbd_congestion_fee",
]

agg_exprs = []
for c in numeric_cols:
    agg_exprs.append(F.min(c).alias(f"{c}__min"))
    agg_exprs.append(F.max(c).alias(f"{c}__max"))

ranges_row = df.agg(*agg_exprs).first().asDict()

print(f"{'Columna':<25} {'min':>20} {'max':>20}")
for c in numeric_cols:
    mn = ranges_row[f"{c}__min"]
    mx = ranges_row[f"{c}__max"]
    print(f"{c:<25} {str(mn):>20} {str(mx):>20}")

### Lectura del bloque 5.2

La tabla muestra los valores mínimo y máximo observados en cada columna numérica del dataset completo. Se compara cada rango contra el rango del tipo destino propuesto:

| Columna | Tipo destino propuesto | Rango del tipo | Verificación |
|---|---|---|---|
| `passenger_count` | `byte` (8 bits) | -128 a 127 | min ≥ 0 y max ≤ 127 |
| `PULocationID`, `DOLocationID` | `short` (16 bits) | -32 768 a 32 767 | min ≥ 1 y max ≤ 32 767 |
| `trip_distance` | `float` (32 bits) | ~7 dígitos significativos | max menor a 1×10⁷ para preservar precisión decimal |
| Montos en USD (`fare_amount`, `tip_amount`, `total_amount`, etc.) | `float` (32 bits) | ~7 dígitos significativos | max menor a 1×10⁷ |

Si algún max excede el rango del tipo destino, ese tipo se descarta para esa columna o se trata como outlier en la etapa de calidad. Los valores negativos en montos o distancias también se registran: pueden ser anulaciones legítimas (con `RatecodeID` o `payment_type` asociados) o ruido a limpiar.

In [ ]:
# 5.3 Verificación de cbd_congestion_fee tras mergeSchema
print("Columnas en df:")
print(df.columns)
print()

cbd_present = "cbd_congestion_fee" in df.columns
print(f"cbd_congestion_fee presente en el esquema: {cbd_present}")

if cbd_present:
    null_by_year = (
        df.withColumn("year", F.year("tpep_pickup_datetime"))
          .groupBy("year")
          .agg(
              F.count("*").alias("n_filas"),
              F.sum(F.col("cbd_congestion_fee").isNull().cast("int")).alias("n_nulos_cbd"),
          )
          .orderBy("year")
    )
    null_by_year.show(truncate=False)

### Lectura del bloque 5.3

Se confirma que `mergeSchema=True` rescató la columna `cbd_congestion_fee` y se cuantifica la proporción de nulos por año:

- Para `year = 2024`: se espera `n_nulos_cbd = n_filas` (100% de nulos). La columna no existía en los archivos de 2024 y `mergeSchema` la introduce vacía.
- Para `year = 2025`: se esperan algunos nulos durante los primeros días de enero (la tarifa entró en vigor el 5 de enero) y luego mayoría con valor.
- Cualquier `year` distinto a 2024 o 2025 indica fechas fuera del período del dataset, lo cual es un problema de calidad a registrar en el paso siguiente.

## 6. Refinamiento del esquema con downcast de tipos

Las validaciones de la sección 5 confirmaron que los rangos efectivos del dataset caben en tipos más estrechos que los que NYC TLC usa en el Parquet original. Esta sección aplica los downcasts en una sola transformación con `selectExpr` (siguiendo el patrón usado en clases `Ejemplo2.ipynb`), reasignando `df` en sitio.

El motivo del refinamiento no es solamente cosmético. Aunque el archivo Parquet en disco no se modifica, los tipos que Spark expone en su DataFrame se reflejan directamente en la representación interna en memoria: cada fila ocupa menos bytes cuando se materializa. Esto reduce el costo de las operaciones que mueven datos: shuffles entre etapas, broadcast joins, cache en memoria, y la presión sobre la heap del executor. Para un dataset de ~90 millones de filas, el ahorro agregado es relevante.

Se reasigna `df` en sitio para evitar tener dos referencias al mismo plan lógico ocupando espacio mental: el `df` post-cast es la única fuente de verdad para las etapas siguientes.

### Plan de casts y ahorro estimado

| Columna | Tipo actual (bytes) | Tipo destino (bytes) | Bytes ahorrados / fila | Justificación |
|---|---|---|---|---|
| `VendorID` | int (4) | tinyint (1) | 3 | Enum oficial {1, 2, 6, 7}; max=7 ≤ 127 |
| `passenger_count` | long (8) | tinyint (1) | 7 | Rango observado [0, 9]; max=9 ≤ 127 |
| `RatecodeID` | long (8) | tinyint (1) | 7 | Enum {1-6, 99}; max=99 ≤ 127 |
| `PULocationID` | int (4) | smallint (2) | 2 | Rango [1, 265]; cabe en short [-32k, 32k] |
| `DOLocationID` | int (4) | smallint (2) | 2 | Igual que `PULocationID` |
| `payment_type` | long (8) | tinyint (1) | 7 | Enum {0-6}; max observado=5 ≤ 127 |
| `trip_distance` | double (8) | float (4) | 4 | Distancias en millas; precisión float (~7 dígitos significativos) suficiente para valores reales (<10⁴ mi). Outliers (398k mi) son bogus y se filtrarán en limpieza |
| `fare_amount` | double (8) | float (4) | 4 | Tarifas típicas <USD 1000; float preserva centavos hasta ~USD 10⁵ |
| `extra` | double (8) | float (4) | 4 | Recargos pequeños (<USD 200) |
| `mta_tax` | double (8) | float (4) | 4 | Valores pequeños |
| `tip_amount` | double (8) | float (4) | 4 | Propinas observadas <USD 1000 |
| `tolls_amount` | double (8) | float (4) | 4 | Peajes observados <USD 2000 |
| `improvement_surcharge` | double (8) | float (4) | 4 | Recargo casi fijo |
| `total_amount` | double (8) | float (4) | 4 | Mismo argumento que `fare_amount` |
| `congestion_surcharge` | double (8) | float (4) | 4 | Recargo casi fijo (<USD 5) |
| `Airport_fee` | double (8) | float (4) | 4 | Cargo fijo (<USD 10) |
| `cbd_congestion_fee` | double (8) | float (4) | 4 | Cargo fijo (<USD 5) |

**Total ahorro: 72 bytes / fila** sobre las columnas tipadas.

Sobre 89,892,322 filas, el ahorro agregado es de aproximadamente **6.5 GB en presión de memoria** al materializar el DataFrame (shuffles, cache, broadcast, plan execution). El tamaño en disco del Parquet original no cambia: el ahorro se realiza en tiempo de ejecución, donde Spark deserializa cada fila a su representación interna (Catalyst InternalRow) y los tipos más estrechos ocupan menos memoria por fila.

**Columnas que no se modifican**:

| Columna | Tipo | Razón |
|---|---|---|
| `tpep_pickup_datetime` | timestamp_ntz | Ya es el tipo correcto (sin zona horaria, según convención NYC TLC) |
| `tpep_dropoff_datetime` | timestamp_ntz | Igual |
| `store_and_fwd_flag` | string | Solo 3 valores ('Y', 'N', null); convertir a boolean perdería la categoría null. Se evaluará en la etapa de limpieza |

In [ ]:
# Aplicación de los casts en una sola transformación con selectExpr.
# Patrón inspirado en sem2/classroom/Ejemplo2.ipynb (DDL inline en cada expresión).
# Se reasigna `df` en sitio: la versión refinada reemplaza a la original, no se duplica el dataset.
df = df.selectExpr(
    "cast(VendorID as tinyint) VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "cast(passenger_count as tinyint) passenger_count",
    "cast(trip_distance as float) trip_distance",
    "cast(RatecodeID as tinyint) RatecodeID",
    "store_and_fwd_flag",
    "cast(PULocationID as smallint) PULocationID",
    "cast(DOLocationID as smallint) DOLocationID",
    "cast(payment_type as tinyint) payment_type",
    "cast(fare_amount as float) fare_amount",
    "cast(extra as float) extra",
    "cast(mta_tax as float) mta_tax",
    "cast(tip_amount as float) tip_amount",
    "cast(tolls_amount as float) tolls_amount",
    "cast(improvement_surcharge as float) improvement_surcharge",
    "cast(total_amount as float) total_amount",
    "cast(congestion_surcharge as float) congestion_surcharge",
    "cast(Airport_fee as float) Airport_fee",
    "cast(cbd_congestion_fee as float) cbd_congestion_fee",
)

print("Esquema refinado:")
df.printSchema()

### Esquema refinado

`df` queda reasignado en sitio con los tipos definitivos para el resto del análisis. Las próximas etapas (análisis exploratorio, calidad de datos, muestreo y modelado) operan sobre este esquema.

`df` apunta ahora a un plan lógico que incluye la lectura del Parquet original más las transformaciones de cast. Spark sigue leyendo los archivos en su tipo nativo desde disco y aplica los downcasts en memoria en cada acción. Si en una etapa posterior se requiere persistir el dataset refinado en disco para acelerar iteraciones, se puede escribir como Parquet con el esquema nuevo (`df.write.parquet("data/processed/")`); por ahora no es necesario.